# Sophia AGI — LoRA training (Google Colab)

Train **Sophia** adapter on the provenance corpus with QLoRA (4-bit).

**Runtime:** GPU (T4 works for 3B; L4/A100 for 7B)

**Repo:** [github.com/tomyimkc/sophia-agi](https://github.com/tomyimkc/sophia-agi)

In [ ]:
!git clone https://github.com/tomyimkc/sophia-agi.git
%cd sophia-agi

import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q -r requirements-lora.txt bitsandbytes accelerate

In [ ]:
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL = "Qwen/Qwen2.5-7B-Instruct" if vram_gb > 20 else "Qwen/Qwen2.5-3B-Instruct"
print(f"Using {MODEL} ({vram_gb:.1f} GB VRAM)")

In [ ]:
!python tools/prepare_lora_dataset.py
!python tools/train_lora.py --4bit --epochs 3 --model {MODEL} --batch-size 2

In [ ]:
!python tools/claude_model_lab.py write-modelfile --adapter training/lora/checkpoints/sophia-v1

import shutil
from google.colab import files

shutil.make_archive("sophia-lora-v1", "zip", "training/lora/checkpoints/sophia-v1")
files.download("sophia-lora-v1.zip")

## After download (local)

```bash
unzip sophia-lora-v1.zip -d training/lora/checkpoints/sophia-v1
python tools/eval_local_model.py --adapter training/lora/checkpoints/sophia-v1 --with-gate
ollama create sophia-7b -f models/ollama/Modelfile
```

Always run **sophia_gate_check** on production answers.